### 외부 데이터 개요
1. 데이터셋 이름 : 서울특별시 강동구_코로나현황(구 단위)_20230901.csv
2. 데이터 셋 출처 : https://www.data.go.kr/data/15122054/fileData.do
3. 데이터 선택 이유 : 관심 도메인이 의료보건, 이커머스인데 이커머스 데이터는 LMS에서 학습한 것 같아, 의료보건으로 선택함
4. 각 행의 의미 : 일자별 강동구 관내에서 발생한 코로나바이러스 감염의 확진자 발생 및 감염 관리 현황
5. 주요 칼럼의 의미 : 칼럼명과 동일

In [1]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 불러오기 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
!pip install numpy pandas matplotlib seaborn pyarrow -q

import os
import platform
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")   # 학습 중 경고 메시지를 잠시 숨깁니다.

# 재현성: 같은 난수를 항상 같게 만들어 결과가 매번 동일하도록 고정합니다.
np.random.seed(42)

# 한글 폰트 설정 (그래프 안 글자가 깨지지 않도록 운영체제별로 분기)
system = platform.system()
if system == "Darwin":          # macOS
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":       # Windows
    plt.rcParams["font.family"] = "Malgun Gothic"
else:                            # Linux 등
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

# 결과 저장용 임시 폴더 (이 노트북 옆에 'd009_outputs/' 가 만들어집니다)
OUT_DIR = Path("d009_outputs")
OUT_DIR.mkdir(exist_ok=True)

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)
print("저장 폴더:", OUT_DIR.resolve())

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.4.6
pandas: 3.0.3
저장 폴더: C:\DI\D009\d009_outputs


In [11]:
# ─────────────────────────────────────────────
# 데이터 불러오기 — 서울 강동구 코로나 현황 CSV
# ─────────────────────────────────────────────
file_path = r"C:\DI\D009\서울특별시 강동구_코로나현황(구 단위)_20230901.csv"
#file_path = "서울특별시_강동구_코로나현황_구_단위__20230901__1_.csv"

print(repr(file_path))
# 한국 공공데이터는 보통 cp949(EUC-KR) 인코딩인 경우가 많아서 먼저 시도해보고,
# 안 되면 utf-8로 재시도합니다.
try:
    covid = pd.read_csv(file_path, encoding="cp949")
except UnicodeDecodeError:
    covid = pd.read_csv(file_path, encoding="utf-8")

print("shape:", covid.shape)
covid.head()

'C:\\DI\\D009\\서울특별시 강동구_코로나현황(구 단위)_20230901.csv'
shape: (1281, 8)


,기준일,기준시,신규확진자 수,누적확진자 수,신규완치자 수,누적완치자 수,사망자 수,자가격리인원
0,2020-02-07,18시,0.0,0,0,0,NaN,15.0
1,2020-02-08,18시,0.0,0,0,0,NaN,15.0
2,2020-02-09,18시,0.0,0,0,0,NaN,11.0
3,2020-02-10,18시,0.0,0,0,0,NaN,12.0
4,2020-02-11,18시,0.0,0,0,0,NaN,8.0


In [12]:
# 품질 리포트 함수 v1 — 결측·중복·타입을 한 번에 보여주는 진단기
def quality_report(df: pd.DataFrame, name: str = "df") -> pd.DataFrame:
    '''데이터프레임의 품질을 컬럼별로 진단해 한 표로 돌려줍니다.'''
    n_rows = len(df)
    report = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True),
        "sample": [df[col].dropna().iloc[0] if df[col].notna().any() else None for col in df.columns],
    })
    print(f"[품질 리포트] {name}")
    print(f"  행 수: {n_rows:,}  /  열 수: {len(df.columns)}")
    print(f"  메모리: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
    print(f"  완전 중복 행: {df.duplicated().sum()}건")
    return report

In [13]:
# 품질 리포트 함수 v2 — 수치형 컬럼에 IQR 이상치 비율을 추가
def quality_report_full(df: pd.DataFrame, name: str = "df") -> pd.DataFrame:
    '''v1에 수치형 이상치 비율(IQR)과 의심 타입 컬럼 표시를 추가합니다.'''
    n_rows = len(df)
    base = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True),
    })

    # IQR 이상치 비율 (수치형 컬럼만)
    outlier_pct = {}
    for col in df.select_dtypes(include="number").columns:
        s = df[col].dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        outlier_pct[col] = ((s < lo) | (s > hi)).mean() * 100
    base["outlier_pct_iqr"] = pd.Series(outlier_pct).round(2)

    # object 컬럼이 실제로는 날짜로 파싱되는지 의심 표시
    suspicious_datetime = []
    for col in df.select_dtypes(include="object").columns:
        try:
            parsed = pd.to_datetime(df[col], errors="coerce")
            if parsed.notna().mean() > 0.8:
                suspicious_datetime.append(col)
        except Exception:
            pass
    base["maybe_datetime"] = base.index.isin(suspicious_datetime)

    print(f"[품질 리포트(완전판)] {name}")
    print(f"  행 수: {n_rows:,}  /  열 수: {len(df.columns)}")
    print(f"  완전 중복 행: {df.duplicated().sum()}건")
    if suspicious_datetime:
        print(f"  📌 날짜로 보이는 object 컬럼: {suspicious_datetime}")
    return base


In [14]:
# 시나리오1 - 품질 질단
report_v1 = quality_report_full(covid, name="covid")
report_v1

# 결측치가 가장 높은 칼럼 : 자가격리인원 (결측 534건, 41.69%)
# maybe_datetime이 True로 잡힌 컬럼 : 기준일
# IQR 이상치 비율이 가장 높은 수치형 컬럼 : 신규확진자 수(13.21%), 신규완치자 수(7.57%), 자가격리인원(2.95%) 순

[품질 리포트(완전판)] covid
  행 수: 1,281  /  열 수: 8
  완전 중복 행: 13건
  📌 날짜로 보이는 object 컬럼: ['기준일']


,dtype,missing,missing_pct,n_unique,outlier_pct_iqr,maybe_datetime
기준일,str,0,0.00,1214,NaN,True
기준시,str,0,0.00,5,NaN,False
신규확진자 수,float64,2,0.16,452,13.21,False
누적확진자 수,str,0,0.00,1056,NaN,False
신규완치자 수,int64,0,0.00,25,7.57,False
누적완치자 수,str,27,2.11,681,NaN,False
사망자 수,float64,310,24.20,151,0.00,False
자가격리인원,float64,534,41.69,458,2.95,False


---
### 정제 계획

[1] 중복 제거
    → 완전 중복 13건 제거 (모든 통계의 기준점이 흔들리지 않게 가장 먼저)

[2] 문자열 정제
    → 기준시 오타/표기 통일: '18', '18tl' → '18시' (실질적으로 '18시'/'10시'/'17시' 3개 범주로 정리)
    → 누적확진자 수, 누적완치자 수: object(str) → int 타입 변환
      (콤마 없는 순수 숫자 문자열이라 astype(int)만으로 해결)

[3] 날짜 파싱
    → 기준일: str → datetime 변환 (maybe_datetime=True로 잡혔던 컬럼)
    → 기준시는 datetime으로 바꾸지 않고 범주형(문자열)로 유지
      (측정 시각을 나타내는 레이블일 뿐, 날짜와 합쳐 하나의 timestamp를 만들 필요는 없다고 판단)
    → 기준일 기준 오름차순 정렬

[4] 이상치 처리
    → 신규확진자 수(IQR 이상치 13.21%) 등은 삭제하지 않고 별도 플래그(is_outlier)로 표시
      (집단감염 등 실제 현상일 가능성이 높아 제거하면 안 된다고 판단)

[5] 결측치 처리
    → 사망자 수(24.2%), 자가격리인원(41.69%), 누적완치자 수(2.11%) 결측 처리
    → 위 단계(특히 [3] 날짜순 정렬) 이후, 결측이 초기 시기에 몰려있는지 먼저 시각화로 확인
    → 시기별 패턴에 따라 처리 방법 결정 (일괄 채우기보다 구간별 판단)
---

In [18]:
# 시나리오 2 — 정제 파이프라인


def c_dedup(df):
    return df.drop_duplicates().reset_index(drop=True)  # [1] 완전 중복 제거

def c_fix_time(df):
    return df.assign(  # [2] 기준시 표기 통일
        기준시=df["기준시"].replace({"18": "18시", "18tl": "18시"})
    )

def c_fix_numeric_types(df):
    return df.assign(  # [2] str -> 숫자형
        누적확진자수=pd.to_numeric(df["누적확진자 수"], errors="coerce"),
        누적완치자수=pd.to_numeric(df["누적완치자 수"], errors="coerce"),
    ).drop(columns=["누적확진자 수", "누적완치자 수"])

def c_parse_date(df):
    fixed = (df["기준일"].str.strip()
             .str.replace(r"\s+", "-", regex=True)
             .str.replace(r"-+", "-", regex=True))  # 공백/중복하이픈 복구 시도
    return (
        df.assign(기준일=pd.to_datetime(fixed, errors="coerce"))
        .dropna(subset=["기준일"])   # 일자 정보 자체가 없는 복구불가 14건 제거
        .sort_values("기준일")
        .reset_index(drop=True)
    )

def c_flag_outliers(df, col="신규확진자 수"):
    s = df[col]  # [4] IQR 이상치 -> 플래그만
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return df.assign(is_outlier=(s < lo) | (s > hi))

def c_handle_missing(df):
    return df.assign(  # [5] 사망자 수는 누적값이므로 전날 값 이어쓰기가 맞음
        누적사망자수=df["사망자 수"].ffill().fillna(0),
        누적완치자수=df["누적완치자수"].ffill(),
        자가격리인원_측정여부=df["자가격리인원"].notna(),
    ).drop(columns=["사망자 수"])


covid_clean = (
    covid
    .pipe(c_dedup)
    .pipe(c_fix_time)
    .pipe(c_fix_numeric_types)
    .pipe(c_parse_date)
    .pipe(c_flag_outliers)
    .pipe(c_handle_missing)
)

covid_clean["month"] = covid_clean["기준일"].dt.to_period("M").astype(str)
covid_clean["year"] = covid_clean["기준일"].dt.year

print(f"정제 전: {covid.shape}  →  정제 후: {covid_clean.shape}")
print(f"이상치(신규확진자 수) 플래그 건수: {covid_clean['is_outlier'].sum()}건")
print(f"자가격리인원 미측정 건수: {(~covid_clean['자가격리인원_측정여부']).sum()}건")
print(f"기준시 종류: {covid_clean['기준시'].unique()}")

정제 전: (1281, 8)  →  정제 후: (1254, 12)
이상치(신규확진자 수) 플래그 건수: 162건
자가격리인원 미측정 건수: 525건
기준시 종류: <ArrowStringArray>
['18시', '10시', '17시']
Length: 3, dtype: str


In [27]:
# 시나리오 3 — 변환 + Parquet 저장

# ── 계절성 확인용 — 연도 x 월 교차표로 실제 패턴 확인 ──
year_month_pivot = (
    covid_clean.groupby([covid_clean["기준일"].dt.year, covid_clean["기준일"].dt.month])["신규확진자 수"]
    .mean()
    .unstack()
    .round(1)
)
year_month_pivot.index.name = "year"
year_month_pivot.columns.name = "월(1~12월)"
print("연도별 x 월별 평균 신규확진자:")
display(year_month_pivot)

# ── KPI 2: 연도별 비교 — 그 해 신규 사망자 수(연말-연초 누적차이) & 치명률 ──
yearly_kpi = covid_clean.groupby("year").agg(
    신규확진자_합계=("신규확진자 수", "sum"),
    연초_누적사망자=("누적사망자수", "first"),
    연말_누적사망자=("누적사망자수", "last"),
)
yearly_kpi["해당연도_신규사망자"] = yearly_kpi["연말_누적사망자"] - yearly_kpi["연초_누적사망자"]
yearly_kpi["치명률(%)"] = (yearly_kpi["해당연도_신규사망자"] / yearly_kpi["신규확진자_합계"] * 100).round(3)
print("\n연도별 KPI (범주별 비교 = 연도별 비교):")
display(yearly_kpi)

# ── Parquet 저장 ──
year_month_pivot.to_parquet(OUT_DIR / "covid_year_month_pivot.parquet")
yearly_kpi.to_parquet(OUT_DIR / "covid_yearly_kpi.parquet")
print(f"\nParquet 저장 완료: {OUT_DIR.resolve()}")

연도별 x 월별 평균 신규확진자:


월(1~12월),1,2,3,4,5,6,7,8,9,10,11,12
year,,,,,,,,,,,,
2020,NaN,0.2,0.2,0.1,0.4,0.4,0.5,3.0,1.5,0.6,2.3,9.9
2021,5.0,2.9,8.0,8.2,10.1,9.4,17.2,15.0,28.6,32.5,54.9,96.7
2022,77.7,900.0,3267.6,1078.5,188.9,68.6,477.5,938.7,434.6,265.3,508.3,579.8
2023,299.7,105.2,108.0,132.1,198.8,153.6,302.3,376.5,NaN,NaN,NaN,NaN



연도별 KPI (범주별 비교 = 연도별 비교):


,신규확진자_합계,연초_누적사망자,연말_누적사망자,해당연도_신규사망자,치명률(%)
year,,,,,
2020,581.0,0.0,2.0,2.0,0.344
2021,8481.0,2.0,112.0,110.0,1.297
2022,265423.0,115.0,288.0,173.0,0.065
2023,37158.0,289.0,306.0,17.0,0.046



Parquet 저장 완료: C:\DI\D009\d009_outputs
